# Fashion-MNIST Clothing Item Recognition

**Contributor:** Syed Muhammad Imad  

This notebook compares a dense Multi-Layer Perceptron (MLP) and a small Convolutional Neural Network (CNN) on Fashion-MNIST. The full runnable implementation is also available in `src/main.py`.

## CLO-1: Problem Definition

Fashion-MNIST classification is a supervised, multi-class image classification problem. Each input is a normalized `1 x 28 x 28` grayscale clothing image, and each output is one of 10 class labels. Neural networks are suitable because clothing categories require non-linear visual features; a purely linear model cannot reliably separate shirts, coats, pullovers, and other similar shapes in raw pixel space.

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
OUTPUTS = PROJECT_ROOT / 'results'
with open(OUTPUTS / 'metrics.json', 'r', encoding='utf-8') as f:
    metrics = json.load(f)

metrics['metadata']

## CLO-2: Candidate Architectures

- **MLP baseline:** `Flatten -> Linear(784,128) -> ReLU -> Linear(128,64) -> ReLU -> Linear(64,10)`. This is a fast dense baseline, but flattening removes spatial locality.
- **Small CNN:** `Conv(1,16,3,pad=1) -> ReLU -> MaxPool -> Conv(16,32,3,pad=1) -> ReLU -> MaxPool -> Linear(1568,64) -> ReLU -> Linear(64,10)`. This model uses local filters and weight sharing to learn reusable spatial features.

In [ ]:
# The implementation can be inspected directly from the project modules.
from src.models import MLP, SmallCNN, count_params

mlp = MLP()
cnn = SmallCNN()
print('MLP parameters:', count_params(mlp))
print('CNN parameters:', count_params(cnn))

## CLO-3: Training and Evaluation

The experiment was run with PyTorch, CrossEntropyLoss, Adam, batch size 64, and 5 epochs. The original 60,000-image Fashion-MNIST training set was split 90/10 into 54,000 training images and 6,000 validation images. Final metrics were computed on the untouched 10,000-image test set.

To reproduce from scratch, run:

```bash
python src/main.py --epochs 5 --train-subset 60000
```

In [ ]:
rows = []
for model_name, info in metrics['models'].items():
    test = info['test_metrics']
    rows.append({
        'model': model_name,
        'params': info['params'],
        'train_time_sec': round(info['history']['train_time_sec'], 2),
        'accuracy': round(test['accuracy'], 4),
        'precision': round(test['precision'], 4),
        'recall': round(test['recall'], 4),
        'f1': round(test['f1'], 4),
    })

pd.DataFrame(rows)

| Model | Params | Train time | Accuracy | Precision | Recall | F1-score |
|---|---:|---:|---:|---:|---:|---:|
| MLP | 109,386 | 77.00s | 87.26% | 87.39% | 87.26% | 87.29% |
| CNN | 105,866 | 155.67s | 90.67% | 90.78% | 90.67% | 90.67% |

### Training Curves

[Media omitted]

[Media omitted]

### Confusion Matrices

[Media omitted]

[Media omitted]

## CLO-4: Comparative Analysis

The CNN achieved **90.67%** test accuracy compared with **87.26%** for the MLP, a gain of **3.41 percentage points**. It also used slightly fewer trainable parameters (105,866 vs. 109,386), but required more CPU training time (155.67s vs. 77.00s).

The main confusion errors are concentrated in visually similar upper-body classes. The MLP most often confused Shirt with T-shirt/top and Coat with Pullover. The CNN reduced overall error but still confused T-shirt/top, Shirt, Coat, and Pullover because those classes share sleeves, torso shapes, and fabric-like texture regions.

**Conclusion:** The CNN is the better architecture for Fashion-MNIST because convolution preserves local spatial structure and learns reusable visual filters. The MLP is faster and simpler, but flattening the input makes it less effective at distinguishing similar clothing categories.

## References

[1] H. Xiao, K. Rasul, and R. Vollgraf, "Fashion-MNIST: a novel image dataset for benchmarking machine learning algorithms," arXiv:1708.07747, 2017.  
[2] A. Paszke et al., "PyTorch: An imperative style, high-performance deep learning library," Advances in Neural Information Processing Systems, vol. 32, 2019.  
[3] F. Pedregosa et al., "Scikit-learn: Machine learning in Python," Journal of Machine Learning Research, vol. 12, pp. 2825-2830, 2011.